In [0]:
from pyspark.sql import functions as F

In [0]:
# List files in the volume to understand what we're working with
volume_path = "/Volumes/turbine_poc/wind_farm/landing/source_data"

import os
files = dbutils.fs.ls(volume_path)
for file in files:
    print(f"{file.name} - Size: {file.size} bytes")

In [0]:
# Load all 3 CSV files from the volume into a single dataframe
df = spark.read.csv(
    f"{volume_path}/*.csv",
    header=True,
    inferSchema=True
)

# Display basic info about the combined dataframe
print(f"Total rows: {df.count():,}")
print(f"Total columns: {len(df.columns)}")
print("\nSchema:")
df.printSchema()
print("\nFirst few rows:")
display(df.limit(10))

In [0]:
#duplicates
print(df.count())
print(df.dropDuplicates().count())

In [0]:
display(df.describe())

In [0]:
cast_check = df.select(
    F.sum(F.when(F.col("timestamp").isNotNull() &
                 F.to_timestamp("timestamp").isNull(), 1).otherwise(0)).alias("bad_timestamp"),
    F.sum(F.when(F.col("turbine_id").isNotNull() &
                 F.col("turbine_id").cast("int").isNull(), 1).otherwise(0)).alias("bad_turbine_id"),
    F.sum(F.when(F.col("wind_speed").isNotNull() &
                 F.col("wind_speed").cast("double").isNull(), 1).otherwise(0)).alias("bad_wind_speed"),
    F.sum(F.when(F.col("wind_direction").isNotNull() &
                 F.col("wind_direction").cast("int").isNull(), 1).otherwise(0)).alias("bad_wind_direction"),
    F.sum(F.when(F.col("power_output").isNotNull() &
                 F.col("power_output").cast("double").isNull(), 1).otherwise(0)).alias("bad_power_output"),
)
display(cast_check)


In [0]:
# Correlation analysis between wind speed and power output
from pyspark.sql.functions import corr

# Calculate correlation
correlation = df.select(corr("wind_speed", "power_output")).collect()[0][0]
print(f"Correlation between wind_speed and power_output: {correlation:.4f}")

# Show correlation matrix for all numeric columns
print("\nFull correlation analysis:")
for col1 in ["wind_speed", "wind_direction", "power_output"]:
    for col2 in ["wind_speed", "wind_direction", "power_output"]:
        if col1 != col2:
            corr_val = df.select(corr(col1, col2)).collect()[0][0]
            print(f"{col1} vs {col2}: {corr_val:.4f}")

In [0]:
# Create wind-band table to visualize power curve
# Group wind speeds into 1 m/s bands

wind_band_df = df.withColumn(
    "wind_band", 
    F.floor(F.col("wind_speed")).cast("int")
).groupBy("wind_band").agg(
    F.avg("power_output").alias("avg_power_output"),
    F.min("power_output").alias("min_power_output"),
    F.max("power_output").alias("max_power_output"),
    F.stddev("power_output").alias("stddev_power_output"),
    F.count("*").alias("observation_count")
).orderBy("wind_band")

print("Wind-Band Power Curve Table:")
display(wind_band_df)

In [0]:
# Visualize the power curve
import matplotlib.pyplot as plt

# Convert to pandas for plotting
wind_band_pd = wind_band_df.toPandas()

plt.figure(figsize=(12, 6))

# Plot average power output with error bars (showing min/max range)
plt.subplot(1, 2, 1)
plt.plot(wind_band_pd['wind_band'], wind_band_pd['avg_power_output'], 'o-', linewidth=2, markersize=8)
plt.fill_between(wind_band_pd['wind_band'], 
                  wind_band_pd['min_power_output'], 
                  wind_band_pd['max_power_output'], 
                  alpha=0.3)
plt.xlabel('Wind Speed (m/s)', fontsize=12)
plt.ylabel('Power Output (MW)', fontsize=12)
plt.title('Wind Turbine Power Curve', fontsize=14, fontweight='bold')
plt.grid(True, alpha=0.3)

# Plot observation counts per wind band
plt.subplot(1, 2, 2)
plt.bar(wind_band_pd['wind_band'], wind_band_pd['observation_count'], color='steelblue')
plt.xlabel('Wind Speed (m/s)', fontsize=12)
plt.ylabel('Number of Observations', fontsize=12)
plt.title('Data Distribution by Wind Speed', fontsize=14, fontweight='bold')
plt.grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.show()

print("\nPower Curve Analysis:")
print(f"Wind speed range: {wind_band_pd['wind_band'].min()} - {wind_band_pd['wind_band'].max()} m/s")
print(f"Peak power output: {wind_band_pd['avg_power_output'].max():.2f} MW at {wind_band_pd.loc[wind_band_pd['avg_power_output'].idxmax(), 'wind_band']} m/s")

In [0]:
#My next question would be if at the same time all the turbines face the same weather conditions, and generate a similar power output or not, which could be another rule to define, are clusters of turbines with similar conditions at the same timestamps?